In [1]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [2]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [3]:
v1 = model.encode(q1)

This shows how many dimentions 

In [4]:
v1.shape

(384,)

Encode a document

In [5]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

We do the dot product to see how similar they are

In [6]:
v1.dot(dv)

np.float32(0.39572883)

In [7]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

The similariy of this product is zero which mean they are very different 

In [8]:
v2.dot(dv)

np.float32(0.019730574)

In [12]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-23 16:06:20--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     888  --.-KB/s    in 0s      

2026-06-23 16:06:21 (45.8 MB/s) - ‘ingest.py’ saved [888/888]



In [9]:
from ingest import load_faq_data

documents = load_faq_data()

In [10]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

Each document is a Python dictionary with a question and an answer. We embed both together. That way a query can match against the question text and the answer text in our index.

In [11]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [12]:
len(texts)

1350

### Generating embeddings

Now we generate the embeddings. We won't hand the model all of them at once. That takes a long time, and we can't see what's happening inside. Instead we split them into batches.

In [13]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [14]:
vectors[10]

array([-8.04887991e-03, -4.23095562e-02,  6.89805439e-03,  6.72613755e-02,
        1.07818441e-02, -1.00610405e-01,  1.81949940e-02, -4.78682630e-02,
       -1.01894625e-01,  5.02834171e-02, -1.38202179e-02, -3.17115076e-02,
        1.26795014e-02,  9.05294064e-03, -6.50629550e-02,  2.80794520e-02,
        2.61346102e-02, -1.05766185e-01, -2.94326991e-02, -5.09120710e-02,
        4.69526425e-02, -4.12351973e-02,  5.68785286e-03,  1.17212883e-03,
        5.72158284e-02,  6.57509491e-02,  4.00751829e-02,  4.02278192e-02,
        5.93154579e-02, -7.53384782e-03, -7.75877833e-02,  6.71754330e-02,
        6.38393164e-02,  3.66461053e-02,  1.44551853e-02,  2.51878314e-02,
       -1.48494821e-02, -9.69775766e-02, -4.26857881e-02, -1.43325813e-02,
       -3.55788283e-02,  9.69671309e-02,  5.91354556e-02,  3.31453755e-02,
        1.54764121e-02, -6.20953180e-02, -2.13111211e-02, -4.88513708e-02,
        8.62475950e-03,  1.04914397e-01, -3.21247173e-03, -5.28438315e-02,
       -5.83541952e-02, -

In [15]:
vectors[10].shape

(384,)

We turn them into a 2-dimensional array (matrix) where

- rows are documents (vectors)
- columns are dimensions of the vectors

In [16]:
import numpy as np
X = np.array(vectors)

number of documents vs number of dimensions

In [17]:
X.shape

(1350, 384)

When a query comes in, we embed it:

In [18]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

Next, we compute the dot product against all documents:

In [19]:
scores = X.dot(v_query)

In [20]:
scores

array([ 0.48740587,  0.2099193 ,  0.762941  , ..., -0.08637964,
        0.03759784, -0.03037035], shape=(1350,), dtype=float32)

argmax returns the index of the heighest score

In [21]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.762941))

In [22]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

```np.argsort(scores)``` evaluates the array and returns an array of indices corresponding to the values sorted from lowest to highest.

In [23]:
np.argsort(scores)

array([1089,  217,   77, ...,  907,  625,    2], shape=(1350,))

These are the top 5 indices

In [24]:
top5 = np.argsort(scores)[-5:]

In [26]:
top5

array([  7, 538, 907, 625,   2])

If you need to retrieve the actual highest scores rather than just their index positions, you can index the original scores array with your top5 result

In [27]:
top5_scores = scores[top5]

In [28]:
top5_scores

array([0.56009984, 0.6536311 , 0.7192131 , 0.7579372 , 0.762941  ],
      dtype=float32)

They come out smallest-first, so we reverse them to get the highest first

In [29]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

Now we can read off the top 5 scores:

In [30]:
scores[top5]

array([0.762941  , 0.7579372 , 0.7192131 , 0.6536311 , 0.56009984],
      dtype=float32)

Let's read  the actual documents

In [31]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 

Note: based on the data above, we need to filter by course

different way of doing it

In [32]:
top5 = np.argsort(-scores)[:5]

In [33]:
top5

array([  2, 625, 907, 538,   7])

### Vector Search with minsearch

- `fit` to index data
- `search` to query
- `filter_dict` in search to filter by keyword



##### Mapping Text to Vector Space
- The `X` Array: This contains the mathematical coordinates (embeddings) for the documents.
- The `documents` List: This holds the actual text metadata (the payload).
- The Connection: Indexing links each vector in X to its corresponding document text in documents. When a future search finds the closest mathematical vector, the system knows exactly which text payload to return to the user.

In [25]:
from minsearch import VectorSearch


# its computing the dot product for all the documents we have 
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [26]:
# X = np.array(vectors)
X

array([[-0.02670618, -0.12245757,  0.01594413, ..., -0.00230654,
        -0.11218394, -0.02365559],
       [-0.01099552, -0.11074744, -0.02536942, ...,  0.09022228,
        -0.02697371,  0.01975672],
       [-0.08896548, -0.06128178,  0.00775603, ...,  0.0405971 ,
         0.00479277, -0.02745943],
       ...,
       [-0.03652925,  0.01415426, -0.06838644, ...,  0.04316786,
         0.08105537, -0.02148626],
       [-0.13091588, -0.06990605, -0.0093188 , ..., -0.00044342,
        -0.0128573 ,  0.01426918],
       [-0.07984784,  0.01926981,  0.02544978, ..., -0.03368027,
        -0.01884026,  0.05837054]], shape=(1350, 384), dtype=float32)

In [27]:
v1

array([-7.94797949e-03, -9.18932408e-02, -1.14074284e-02,  2.18466595e-02,
        1.11858333e-02, -1.30818449e-02, -7.39962235e-02, -9.87466797e-02,
       -1.05911419e-01, -3.03381458e-02, -2.92174686e-02,  2.33883355e-02,
        7.87481945e-03,  4.15800288e-02,  2.42720488e-02, -3.65723185e-02,
       -5.31095453e-02, -1.94869116e-02, -2.26979833e-02,  8.76777805e-03,
       -1.10785306e-01,  3.97906862e-02, -4.18479182e-02,  2.82960795e-02,
       -1.28302518e-02,  1.72567349e-02,  3.10428441e-02,  9.51102898e-02,
       -1.90717615e-02, -6.23236150e-02, -3.59101109e-02,  6.46180734e-02,
        3.06465179e-02,  2.39972025e-02,  2.06126980e-02,  9.87384096e-03,
        7.63835683e-02, -6.50510937e-02,  4.07937029e-03,  2.34527756e-02,
       -2.49407142e-02, -2.95020584e-02, -1.74879059e-02,  4.62139063e-02,
        2.48923320e-02,  1.08981736e-01, -5.67837544e-02, -6.95324540e-02,
        4.35405597e-03,  2.85132304e-02,  2.75795385e-02, -2.49844193e-02,
       -6.82148617e-03,  

In [28]:
vindex.search(v1, num_results=5)

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.',
  'doc_id': '41aabbd7c5'},
 {'course': 'mlops-zoomcamp',
  'section': 'General Cour

In [29]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [40]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-24 13:24:15--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-24 13:24:15 (11.6 MB/s) - ‘rag_helper.py’ saved [2134/2134]



### RAG with Vector Search

In [41]:
from rag_helper import RAGBase



In [42]:
from gem_client import call_gemini, init

# Initialize the .env file auto-detection on load
init()

In [62]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [63]:
# Define the unified class right here in your notebook cell
class RAGVectorGemini(RAGBase):

    def __init__(self, index, embedder, instructions=INSTRUCTIONS, prompt_template=PROMPT_TEMPLATE, course='llm-zoomcamp', model='gemini-2.5-flash'):
        # Fallback to the parent class defaults if instructions/templates aren't passed
        super().__init__(
            index=index,
            instructions=instructions,
            prompt_template=prompt_template,
            course=course,
            model=model,
            llm_client=None
        )
        self.embedder = embedder

    def search(self, query, num_results=5):
        """Overrides parent keyword search to use your sentence embedder."""
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

    def llm(self, prompt):
        """Overrides parent OpenAI payload to use your local Gemini wrapper."""
        # Use self.instructions if populated, otherwise fallback to base RAGBase strings
        instructions_str = self.instructions if self.instructions else "Your task is to answer questions from the course participants based on the provided context."
        full_prompt = f"{instructions_str}\n\n{prompt}"

        # Route safely through your rate-limiting file wrapper
        result = call_gemini(
            prompt=full_prompt,
            model=self.model,
            max_tokens=1000 # Adjust this token size limit as needed for your RAG answers
        )
        return result.content


In [64]:
vector_assistant = RAGVectorGemini(
    embedder=model,          # Your text embedder model (e.g., SentenceTransformer)
    index=vindex,            # Your vector database index
    model='gemini-2.5-flash', # Explicitly targeting the Gemini model
)

In [65]:
query = "What are the system prerequisites for this course?"
answer = vector_assistant.rag(query)

In [66]:
answer

'If you choose to run the course locally instead of using Codespaces, you will need to set up Python, `uv`, Jupyter, Docker, and any other tools needed for the specific module.'

In [68]:
query = "I just found out about the program, can I still sign up?"
answer = vector_assistant.rag(query)

In [69]:
answer

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [70]:
answer = vector_assistant.rag("the program has already begun, can I still sign up?")

In [71]:
answer

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'